# Análisis LLM-as-a-Judge — Grupo 14 (IIC3633)

Consolida los 10 archivos `judge_output_path` (5 métodos × 2 datasets) generados por `llm_judge.ipynb` (uno por integrante) y produce:
1. Tabla cuantitativa de promedios por método/dataset/métrica (va directo al paper).
2. Análisis cualitativo: peores casos, contraste automático mejor-vs-peor método, taxonomía de patrones de fallo.

**Requisito:** completa `RUNS` abajo con los 10 `judge_output_path` reales (uno por método-dataset), y ajusta `JUDGE_K` al N final acordado por el equipo.

**No corre ninguna llamada a la API** — todo opera sobre los JSON ya guardados.

## 1. Setup

In [1]:
import os, json
import pandas as pd
pd.set_option("display.max_colwidth", None)

In [11]:
from google.colab import  drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/Proyecto RecSys/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Config

In [12]:

JUDGE_K = 60

RUNS = [
    # pearl
    {"method": "cot",        "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/cot_pearl_judge.json"},
    {"method": "reflexion",  "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/reflexion_pearl_judge.json"},
    {"method": "dc",         "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/dc_pearl_judge.json"},
    {"method": "ace_base",   "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/ace_base_pearl_judge.json"},
    {"method": "ace_tools",  "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/ace_tools_pearl_judge.json"},
    # redial
    {"method": "cot",        "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/cot_redial_judge.json"},
    {"method": "reflexion",  "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/reflexion_redial_judge.json"},
    {"method": "dc",         "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/dc_redial_judge.json"},
    {"method": "ace_base",   "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/ace_base_redial_judge.json"},
    {"method": "ace_tools",  "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/ace_tools_redial_judge.json"},
]

## 3. Carga con truncado a K

Se toman los primeros `k` registros guardados en cada archivo.

In [21]:
def load_scores(path, k=JUDGE_K):
    if not os.path.exists(path):
        print(f"⚠️ No existe: {path}")
        return pd.DataFrame()
    df = pd.DataFrame(json.load(open(path)))
    if df.empty:
        return df
    return df.head(k).reset_index(drop=True)

run_lookup = {(r["method"], r["dataset"]): r for r in RUNS}

## 4. Cuantitativo — tabla de promedios (va al paper)

In [24]:
def summarize_all_runs(runs, k=JUDGE_K, save_csv=PROJECT_DIR+"judge/summary_scores.csv"):
    rows = []
    for run_cfg in runs:
        method, dataset = run_cfg["method"], run_cfg["dataset"]
        df = load_scores(run_cfg["judge_output_path"], k=k)
        if df.empty:
            continue
        n = len(df)
        rows.append({
            "method": method,
            "dataset": dataset,
            "n": n,
            "relevance_mean": df["relevance_score"].mean(),
            "relevance_std": df["relevance_score"].std(),
            "coherence_mean": df["coherence_score"].mean(),
            "coherence_std": df["coherence_score"].std(),
            "explainability_mean": df["explainability_score"].mean(),
            "explainability_std": df["explainability_score"].std(),
        })
    summary = pd.DataFrame(rows).sort_values(["dataset", "method"]).reset_index(drop=True)
    os.makedirs(os.path.dirname(save_csv) or ".", exist_ok=True)
    summary.to_csv(save_csv, index=False)
    print(f"Guardado en {save_csv}")
    return summary

In [25]:
summary_df = summarize_all_runs(RUNS)
summary_df

Guardado en /content/drive/MyDrive/Proyecto RecSys/judge/summary_scores.csv


,method,dataset,n,relevance_mean,relevance_std,coherence_mean,coherence_std,explainability_mean,explainability_std
0,ace_base,pearl,60,4.483333,0.999859,4.583333,0.907439,4.733333,0.860955
1,ace_tools,pearl,57,4.421053,0.962648,4.561404,0.707550,4.771930,0.707550
2,cot,pearl,60,4.150000,1.400061,4.300000,1.279566,4.433333,1.240102
3,dc,pearl,58,4.275862,1.253670,4.413793,1.124441,4.603448,1.041922
4,reflexion,pearl,38,4.105263,1.351465,4.421053,0.976247,4.763158,0.714113
5,ace_base,redial,46,2.565217,1.796938,2.326087,1.415067,4.021739,1.164076
6,ace_tools,redial,49,3.367347,1.654462,2.530612,1.209374,3.714286,1.172604
7,cot,redial,60,3.516667,1.523729,4.216667,1.328841,3.916667,1.292635
8,dc,redial,58,3.793103,1.239109,2.465517,1.173023,3.551724,1.062477
9,reflexion,redial,60,4.050000,1.141290,2.583333,1.211410,3.550000,1.294382


In [26]:
# Formato tabla pivotada para el paper
pivot = summary_df.pivot(index="method", columns="dataset",
                          values=["relevance_mean", "coherence_mean", "explainability_mean"])
pivot

relevance_mean           coherence_mean            \
dataset            pearl    redial          pearl    redial   
method                                                        
ace_base        4.483333  2.565217       4.583333  2.326087   
ace_tools       4.421053  3.367347       4.561404  2.530612   
cot             4.150000  3.516667       4.300000  4.216667   
dc              4.275862  3.793103       4.413793  2.465517   
reflexion       4.105263  4.050000       4.421053  2.583333   

          explainability_mean            
dataset                 pearl    redial  
method                                   
ace_base             4.733333  4.021739  
ace_tools            4.771930  3.714286  
cot                  4.433333  3.916667  
dc                   4.603448  3.551724  
reflexion            4.763158  3.550000

## 5. Peores casos por run

In [27]:
def worst_cases(run_cfg, metric="relevance_score", n=3, k=JUDGE_K):
    df = load_scores(run_cfg["judge_output_path"], k=k)
    if df.empty:
        return df
    reasoning_col = metric.replace("_score", "_reasoning")
    return df.sort_values(metric)[["idx", metric, reasoning_col]].head(n)

for run_cfg in RUNS:
    print(f"\n=== {run_cfg['method']}-{run_cfg['dataset']} ===")
    display(worst_cases(run_cfg, metric="relevance_score", n=2))


=== cot-pearl ===


,idx,relevance_score,relevance_reasoning
6,419,1,"The assistant has not yet provided a recommendation in the final message, but since the prompt asks to evaluate the response, and the response is empty, it cannot meet the criteria."
14,2069,1,The assistant has not provided a recommendation in the final message provided for evaluation.



=== reflexion-pearl ===


,idx,relevance_score,relevance_reasoning
2,1126,1,"The user explicitly stated they are not a fan of horror and mystery movies, and The Lighthouse is a psychological horror/mystery film."
6,419,1,"Midsommar is a slow-burn folk horror film, which contradicts the user's explicit request for a 'fast-paced' movie with a 'fusion of horror and action'."



=== dc-pearl ===


...

,idx,relevance_score,relevance_reasoning
2,522,1,"The first recommended title, Coco, was already mentioned by the assistant earlier in the conversation, violating the anti-echo rule."
28,504,1,"The user specifically asked for World War 2 movies and epic historical pieces, but The Fugitive is a 90s crime thriller, not a historical piece."



=== ace_base-redial ===


,idx,relevance_score,relevance_reasoning
1,1,1,"The user explicitly asked for comedies in the last turn and stated they were ready for movie night, but the assistant recommended horror movies instead."
2,2,1,"The first recommended title, The Waterboy (1998), was explicitly mentioned by the user earlier in the conversation, violating the ANTI-ECHO RULE."



=== ace_tools-redial ===


,idx,relevance_score,relevance_reasoning
0,0,1,"The user wanted movies like Super Troopers and American Pie (raunchy/slapstick comedies), but the assistant recommended a serious action thriller."
1,2,1,"The first recommended title, The Waterboy (1998), was explicitly mentioned by the user earlier in the conversation, violating the ANTI-ECHO RULE."
